In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, 1158059f-ac48-4393-ba20-677af96d4f0f, 3, Finished, Available, Finished)

# Vibe Data Engineering

Spark Structured Streaming.  This is likely NOT the best way to do this.  See the last cell for alternative approaches/future.  

* VibeAnalytics LH
* Files/ecommerce/orders-incoming

Prompt:  in the attached lakehouse I have a file called "Files/ecommerce/orders-incoming/olist_orders_dataset-2018.csv".  Load that to a dataframe please

In [1]:
#### ATTENTION: AI-generated code can include errors or operations you didn't intend. Review the code in this cell carefully before running it.

# Define the file path and file name
filePath = "Files/ecommerce/orders-incoming/"
fileName = "olist_orders_dataset-2018.csv"

# Load the CSV file into a Spark DataFrame
spark_df = spark.read.format("csv").option("header", "true").load(filePath + fileName)



StatementMeta(, a639bf75-4ca4-4bfc-b6f3-6c7fb582c1bf, 8, Finished, Available, Finished)

Claude Sonnet 4 prompt:  
Context:
I am tasked with uploading orders information from a company we've just acquired into our lakehouse.  I know nothing about this data.  It is already loaded into a Spark dataframe called "spark_df"

Role:  You are a data engineer with 10 years of experience building data engineering pipelines in spark and spark streaming.  I want you to help me analyze the data using all of your tools,intuition, and experience (Exploratory Data Analytics, etc).  Eventually we are going to "ETL" this data through our lakehouse zones (bronze, silver,gold), but we want to look at the data first.  

Interview:  I want you to help me analyze the data and propose how you would build this solution.  Ask questions along the way, do not make any assumptions about the data.

Task:  Eventually we want to build a spark streaming based solution where another team/process will "drop off" the order files and we will process them.  

# Data Engineering Analysis: Orders Dataset
## Phase 1: Initial Data Exploration

As your data engineer, I'll systematically analyze this acquired company's orders data. Let's start with understanding the basic structure and quality of the data before designing our streaming pipeline.

In [2]:
# Step 1: Basic DataFrame Information
print("=== BASIC DATAFRAME STRUCTURE ===")
print(f"DataFrame Type: {type(spark_df)}")
print(f"Row Count: {spark_df.count():,}")
print(f"Column Count: {len(spark_df.columns)}")
print(f"Columns: {spark_df.columns}")

print("\n=== SCHEMA INFORMATION ===")
spark_df.printSchema()

StatementMeta(, a639bf75-4ca4-4bfc-b6f3-6c7fb582c1bf, 9, Finished, Available, Finished)

=== BASIC DATAFRAME STRUCTURE ===
DataFrame Type: <class 'pyspark.sql.dataframe.DataFrame'>
Row Count: 99,441
Column Count: 8
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

=== SCHEMA INFORMATION ===
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)



In [3]:
# Step 2: Sample Data Inspection
print("=== FIRST 5 ROWS ===")
spark_df.show(5, truncate=False)

print("\n=== RANDOM SAMPLE ===")
spark_df.sample(0.1, seed=42).limit(3).show(truncate=False)

StatementMeta(, a639bf75-4ca4-4bfc-b6f3-6c7fb582c1bf, 10, Finished, Available, Finished)

=== FIRST 5 ROWS ===
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|delivered   |2017-10-02 10:56:33     |2017-10-02 11:07:15|2017-10-04 19:55:00         |2017-10-10 21:25:13          |2017-10-18 00:00:00          |
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|delivered   |2018-07-24 20:41:37     |2018-07-26 03:

In [ ]:
# Step 3: Data Quality Assessment
from pyspark.sql.functions import col, count, when, isnan, isnull, sum as spark_sum

print("=== NULL VALUE ANALYSIS ===")
null_counts = spark_df.select([
    count(when(col(c).isNull() | isnan(col(c)), c)).alias(c) 
    for c in spark_df.columns
])
null_counts.show()

print("\n=== DATA TYPE ASSESSMENT ===")
for col_name, data_type in spark_df.dtypes:
    print(f"{col_name:<25}: {data_type}")
    
print("\n=== ROW-LEVEL DATA COMPLETENESS ===")
total_cols = len(spark_df.columns)
completeness_df = spark_df.withColumn(
    "null_count", 
    sum([when(col(c).isNull() | isnan(col(c)), 1).otherwise(0) for c in spark_df.columns])
).withColumn(
    "completeness_pct", 
    ((total_cols - col("null_count")) / total_cols * 100)
)

completeness_df.select("completeness_pct").describe().show()

StatementMeta(, a639bf75-4ca4-4bfc-b6f3-6c7fb582c1bf, 11, Finished, Available, Finished)

=== NULL VALUE ANALYSIS ===
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+


=== DATA TYPE ASSESSMENT ===
order_id                 : string
customer_id              : string
order_status             : string

In [5]:
# Step 4: Business-Specific Analysis for Orders Data
from pyspark.sql.functions import to_date, year, month, dayofweek, desc

print("=== ORDER PATTERNS ANALYSIS ===")

# Look for date columns that might indicate order timestamps
date_like_columns = [col_name for col_name in spark_df.columns 
                    if any(keyword in col_name.lower() 
                          for keyword in ['date', 'time', 'created', 'order', 'purchase'])]

print(f"Potential date columns found: {date_like_columns}")

# Get unique value counts for key columns (limited to prevent overwhelming output)
print("\n=== KEY COLUMN CARDINALITY ===")
for col_name in spark_df.columns[:10]:  # First 10 columns
    distinct_count = spark_df.select(col_name).distinct().count()
    print(f"{col_name:<25}: {distinct_count:,} unique values")
    
print("\n=== STATISTICAL SUMMARY ===")
spark_df.describe().show()

StatementMeta(, a639bf75-4ca4-4bfc-b6f3-6c7fb582c1bf, 12, Finished, Available, Finished)

=== ORDER PATTERNS ANALYSIS ===
Potential date columns found: ['order_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

=== KEY COLUMN CARDINALITY ===
order_id                 : 99,441 unique values
customer_id              : 99,441 unique values
order_status             : 8 unique values
order_purchase_timestamp : 98,875 unique values
order_approved_at        : 90,734 unique values
order_delivered_carrier_date: 81,019 unique values
order_delivered_customer_date: 95,665 unique values
order_estimated_delivery_date: 459 unique values

=== STATISTICAL SUMMARY ===
+-------+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|summary|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_

In [6]:
# Step 5: Data Anomalies & Quality Issues Detection
print("=== POTENTIAL DATA ISSUES DETECTION ===")

# Check for duplicates
print("Duplicate Analysis:")
total_rows = spark_df.count()
distinct_rows = spark_df.distinct().count()
duplicates = total_rows - distinct_rows
print(f"Total rows: {total_rows:,}")
print(f"Distinct rows: {distinct_rows:,}")
print(f"Potential duplicates: {duplicates:,}")

# Look for empty strings and whitespace issues
print("\n=== STRING DATA QUALITY ===")
string_columns = [col_name for col_name, data_type in spark_df.dtypes if 'string' in data_type.lower()]
print(f"String columns: {string_columns[:5]}...")  # Show first 5

if string_columns:
    empty_string_counts = spark_df.select([
        count(when(col(c) == "", c)).alias(f"{c}_empty") 
        for c in string_columns[:5]  # Check first 5 string columns
    ])
    empty_string_counts.show()

# Check for suspicious patterns
print("\n=== COLUMN VALUE PATTERNS ===")
for col_name in spark_df.columns[:3]:  # First 3 columns for pattern analysis
    print(f"\nColumn: {col_name}")
    spark_df.groupBy(col_name).count().orderBy(desc("count")).limit(5).show()

StatementMeta(, a639bf75-4ca4-4bfc-b6f3-6c7fb582c1bf, 13, Finished, Available, Finished)

=== POTENTIAL DATA ISSUES DETECTION ===
Duplicate Analysis:
Total rows: 99,441
Distinct rows: 99,441
Potential duplicates: 0

=== STRING DATA QUALITY ===
String columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at']...
+--------------+-----------------+------------------+------------------------------+-----------------------+
|order_id_empty|customer_id_empty|order_status_empty|order_purchase_timestamp_empty|order_approved_at_empty|
+--------------+-----------------+------------------+------------------------------+-----------------------+
|             0|                0|                 0|                             0|                      0|
+--------------+-----------------+------------------+------------------------------+-----------------------+


=== COLUMN VALUE PATTERNS ===

Column: order_id
+--------------------+-----+
|            order_id|count|
+--------------------+-----+
|e4de6d53ecff736bc...|    1|
|f373335aac9a659de...| 

## 🔍 Initial Data Engineering Assessment & Questions

**Run the analysis cells above first, then I need to understand:**

### Critical Business Context Questions:
1. **Data Source**: What type of business was this acquired company? (e-commerce, retail, B2B, etc.)
2. **Order Frequency**: How often do you expect new order files to arrive in the streaming scenario?
3. **File Formats**: Will the streaming files always be CSV, or could there be other formats?
4. **Data Volume**: What's the expected daily/hourly volume of orders in the streaming pipeline?
5. **Business Rules**: Are there specific business rules for order validation or processing?

### Technical Architecture Questions:
6. **SLA Requirements**: What are the processing time requirements? (near real-time, hourly, daily)
7. **Error Handling**: How should we handle malformed records, duplicates, or schema changes?
8. **Data Lineage**: Do you need full audit trails of data transformations?
9. **Integration Points**: Will other systems need to consume this data? APIs, dashboards, etc.?

### Lakehouse Design Questions:
10. **Partitioning Strategy**: How should we partition data in bronze/silver/gold layers?
11. **Retention Policies**: How long should we keep data in each layer?
12. **Access Patterns**: Who will be querying this data and how?


# Prompt:

Let me answer some of your questions:

* I'm not too concerned with the missing data (nulls).  I can't control what we're being given.  
* this is a new company we acquired called "NewCo Retail"
* I like your proposed streaming architecture and medallion architecture design. 
* Do not YET worry about setting this code up for streaming.  Let's just get the base Spark code working.   
  * I have no idea what the order volume will be 
  * Processing SLA:  for now, let's not worry about frequency of streaming, let's assume the notebook will be launched manually via a cron job.  We can change this later.  
  * Files will be "dropped" by another process

Let's do this:
* When each new file is delivered let's load that into a schema named "Bronze" and the table should be prefaced with "orders-" and then the name of the file.  This will help us detect future issues with data loading.  Does that sound reasonable?  
* we'll use delta format
* There should be no schema validation.  I am unsure what the future data quality will be.  
* For Silver:  
  * Let's create a new table (if it doesn't exist) in the Silver schema called "Silver.NewCoOrders"
  * convert the string timestamps to proper timestamps
  * I like your partitioning idea, do that
  * Don't worry about NULLs, data quality validations, and rules.  YET!
* For Gold:  
  * Let's create a new table (if it doesn't exist) in the Gold schema called "Gold.OrderMonthlyPKIs"  where you summary the count of the orders for the month grouped by order_status.  

Task:  please modify the notebook accordingly to accomplish these tasks.  


# 🏗️ NewCo Retail - Medallion Architecture Implementation

## Bronze → Silver → Gold Pipeline for Order Data

**Company**: NewCo Retail (Acquired Company)  
**Architecture**: Medallion (Bronze/Silver/Gold) with Delta Lake  
**Processing**: Manual trigger via cron job  
**Data Volume**: TBD - scalable design

## 🥉 Bronze Layer: Raw Data Ingestion

**Purpose**: Ingest raw files into Delta tables with minimal processing  
**Schema**: Bronze.orders-{filename}  
**Format**: Delta Lake  
**Validation**: None (accept all data as-is)  
**Metadata**: Track source file name and ingestion timestamp

In [8]:
# Bronze Layer Implementation
from pyspark.sql.functions import current_timestamp, lit
from datetime import datetime
import os
import re

def load_to_bronze(source_df, filename):
    """
    Load raw data to Bronze layer with metadata tracking
    
    Args:
        source_df: Source DataFrame to load
        filename: Original filename (used for table naming)
    
    Returns:
        None
    """
    
    # Clean filename for table naming (replace hyphens with underscores)
    table_name = os.path.splitext(filename)[0]
    table_name = re.sub(r'[^a-zA-Z0-9_]', '_', table_name)  # Clean special chars
    bronze_table = f"Bronze.orders_{table_name}"
    
    print(f"🥉 Loading to Bronze: {bronze_table}")
    
    # Add metadata columns for data lineage
    bronze_df = source_df.withColumn("source_filename", lit(filename)) \
                         .withColumn("ingestion_timestamp", current_timestamp()) \
                         .withColumn("processing_date", lit(datetime.now().strftime("%Y-%m-%d")))
    
    # Create Bronze schema if it doesn't exist
    spark.sql("CREATE SCHEMA IF NOT EXISTS Bronze")
    
    # Write to Delta table (create if not exists, append new data)
    bronze_df.write \
             .format("delta") \
             .mode("append") \
             .option("mergeSchema", "true") \
             .saveAsTable(bronze_table)
    
    print(f"✅ Successfully loaded {bronze_df.count():,} records to {bronze_table}")
    return bronze_table

# Load current file to Bronze
filename = "olist_orders_dataset-2018.csv"
bronze_table_name = load_to_bronze(spark_df, filename)

StatementMeta(, a639bf75-4ca4-4bfc-b6f3-6c7fb582c1bf, 15, Finished, Available, Finished)

🥉 Loading to Bronze: Bronze.orders_olist_orders_dataset_2018
✅ Successfully loaded 99,441 records to Bronze.orders_olist_orders_dataset_2018


## 🥈 Silver Layer: Cleaned & Transformed Data

**Purpose**: Transform Bronze data into business-ready format  
**Table**: Silver.NewCoOrders  
**Transformations**:  
- Convert string timestamps to proper timestamp types  
- Partition by year/month for performance  
- Standardize data types  
**Strategy**: Incremental load from Bronze tables

In [9]:
# Silver Layer Implementation
from pyspark.sql.functions import to_timestamp, year, month, col, when, isnan, isnull

def transform_to_silver():
    """
    Transform Bronze data to Silver with proper data types and partitioning
    
    Returns:
        Transformed DataFrame ready for Silver layer
    """
    
    print("🥈 Starting Silver layer transformation...")
    
    # Read from all Bronze order tables (future-proof for multiple files)
    bronze_tables = spark.sql("SHOW TABLES IN Bronze").filter(col("tableName").like("orders_%")).collect()
    
    if not bronze_tables:
        print("❌ No Bronze tables found!")
        return None
        
    print(f"📊 Found {len(bronze_tables)} Bronze table(s) to process")
    
    # Union all Bronze order tables
    all_bronze_data = None
    for table_row in bronze_tables:
        table_name = table_row["tableName"]
        print(f"  📝 Reading Bronze.{table_name}")
        
        bronze_df = spark.table(f"Bronze.{table_name}")
        
        if all_bronze_data is None:
            all_bronze_data = bronze_df
        else:
            all_bronze_data = all_bronze_data.union(bronze_df)
    
    # Silver transformations
    silver_df = all_bronze_data.select(
        col("order_id"),
        col("customer_id"), 
        col("order_status"),
        # Convert string timestamps to proper timestamp columns
        to_timestamp(col("order_purchase_timestamp"), "yyyy-MM-dd HH:mm:ss").alias("order_purchase_timestamp"),
        to_timestamp(col("order_approved_at"), "yyyy-MM-dd HH:mm:ss").alias("order_approved_at"),
        to_timestamp(col("order_delivered_carrier_date"), "yyyy-MM-dd HH:mm:ss").alias("order_delivered_carrier_date"),
        to_timestamp(col("order_delivered_customer_date"), "yyyy-MM-dd HH:mm:ss").alias("order_delivered_customer_date"),
        to_timestamp(col("order_estimated_delivery_date"), "yyyy-MM-dd HH:mm:ss").alias("order_estimated_delivery_date"),
        # Keep metadata columns
        col("source_filename"),
        col("ingestion_timestamp"),
        col("processing_date")
    ).withColumn(
        # Add partitioning columns
        "order_year", year(col("order_purchase_timestamp"))
    ).withColumn(
        "order_month", month(col("order_purchase_timestamp"))
    )
    
    print(f"✨ Transformed {silver_df.count():,} records for Silver layer")
    return silver_df

# Create Silver schema and transform data
spark.sql("CREATE SCHEMA IF NOT EXISTS Silver")
silver_df = transform_to_silver()

if silver_df:
    print("🥈 Writing to Silver.NewCoOrders...")
    
    # Write partitioned Delta table
    silver_df.write \
             .format("delta") \
             .mode("overwrite") \
             .partitionBy("order_year", "order_month") \
             .option("overwriteSchema", "true") \
             .saveAsTable("Silver.NewCoOrders")
    
    print("✅ Silver layer complete!")
    
    # Show sample of transformed data
    print("\n📊 Silver Data Sample:")
    spark.table("Silver.NewCoOrders").select(
        "order_id", "order_status", "order_purchase_timestamp", "order_year", "order_month"
    ).show(5)

StatementMeta(, a639bf75-4ca4-4bfc-b6f3-6c7fb582c1bf, 16, Finished, Available, Finished)

🥈 Starting Silver layer transformation...
📊 Found 1 Bronze table(s) to process
  📝 Reading Bronze.orders_olist_orders_dataset_2018
✨ Transformed 99,441 records for Silver layer
🥈 Writing to Silver.NewCoOrders...
✅ Silver layer complete!

📊 Silver Data Sample:
+--------------------+------------+------------------------+----------+-----------+
|            order_id|order_status|order_purchase_timestamp|order_year|order_month|
+--------------------+------------+------------------------+----------+-----------+
|0e6dbbc7484dc1ee0...|    canceled|     2018-03-01 22:40:51|      2018|          3|
|ccd4eb95e68cb64e6...|    canceled|     2018-03-05 22:57:35|      2018|          3|
|7dfff541b2a848ecb...|    canceled|     2018-03-23 13:44:28|      2018|          3|
|7159096c5aa9be77f...|    canceled|     2018-03-15 10:06:35|      2018|          3|
|f5f8998eee8ec7bc5...|    canceled|     2018-03-01 11:42:23|      2018|          3|
+--------------------+------------+------------------------+--------

## 🥇 Gold Layer: Business Metrics & KPIs

**Purpose**: Aggregated business metrics for analytics and reporting  
**Table**: Gold.OrderMonthlyKPIs  
**Metrics**: Monthly order counts grouped by status  
**Use Case**: Executive dashboards, business intelligence, trend analysis

In [ ]:
# Gold Layer Implementation
from pyspark.sql.functions import count, desc, asc, concat, lpad, lit

def create_monthly_kpis():
    """
    Create monthly KPI aggregations from Silver data
    
    Returns:
        DataFrame with monthly order KPIs
    """
    
    print("🥇 Creating Gold layer KPIs...")
    
    # Read from Silver table
    silver_df = spark.table("Silver.NewCoOrders")
    
    # Create monthly KPIs grouped by order status
    monthly_kpis = silver_df.groupBy("order_year", "order_month", "order_status") \
                           .agg(count("order_id").alias("order_count")) \
                           .withColumn("report_period", 
                                     concat(col("order_year").cast("string"), 
                                           lit("-"),
                                           lpad(col("order_month").cast("string"), 2, "0"))) \
                           .orderBy("order_year", "order_month", desc("order_count"))
    
    print(f"📈 Generated KPIs for {monthly_kpis.select('report_period').distinct().count()} months")
    return monthly_kpis

# Create Gold schema and build KPIs
spark.sql("CREATE SCHEMA IF NOT EXISTS Gold")
monthly_kpis_df = create_monthly_kpis()

print("🥇 Writing to Gold.OrderMonthlyKPIs...")

# Write to Gold table
monthly_kpis_df.write \
               .format("delta") \
               .mode("overwrite") \
               .option("overwriteSchema", "true") \
               .saveAsTable("Gold.OrderMonthlyKPIs")

print("✅ Gold layer complete!")

# Show the KPIs
print("\n📊 Monthly Order KPIs by Status:")
spark.table("Gold.OrderMonthlyKPIs").show(20)

StatementMeta(, a639bf75-4ca4-4bfc-b6f3-6c7fb582c1bf, 17, Finished, Available, Finished)

🥇 Creating Gold layer KPIs...
📈 Generated KPIs for 25 months
🥇 Writing to Gold.OrderMonthlyKPIs...
✅ Gold layer complete!

📊 Monthly Order KPIs by Status:
+----------+-----------+------------+-----------+-------------+
|order_year|order_month|order_status|order_count|report_period|
+----------+-----------+------------+-----------+-------------+
|      2016|          9|    canceled|          2|      2016-09|
|      2016|          9|     shipped|          1|      2016-09|
|      2016|          9|   delivered|          1|      2016-09|
|      2016|         10|   delivered|        265|      2016-10|
|      2016|         10|    canceled|         24|      2016-10|
|      2016|         10|    invoiced|         18|      2016-10|
|      2016|         10|     shipped|          8|      2016-10|
|      2016|         10| unavailable|          7|      2016-10|
|      2016|         10|  processing|          2|      2016-10|
|      2016|         12|   delivered|          1|      2016-12|
|      2017| 

## 🎯 Pipeline Summary & Next Steps

### ✅ Medallion Architecture Implemented:

**🥉 Bronze Layer**: `Bronze.orders-{filename}` 
- Raw CSV ingestion with metadata tracking  
- Source filename and ingestion timestamp preserved  
- Delta format for ACID compliance  

**🥈 Silver Layer**: `Silver.NewCoOrders`
- Converted string timestamps to proper timestamp types  
- Partitioned by year/month for query performance  
- Unified view of all order files  

**🥇 Gold Layer**: `Gold.OrderMonthlyKPIs`
- Monthly order counts by status  
- Ready for business intelligence and reporting  

### 🔄 Operational Process:

1. **File Drop**: New order files land in `Files/ecommerce/orders-incoming/`
2. **Cron Trigger**: Notebook runs on schedule  
3. **Bronze Ingestion**: Raw file → `Bronze.orders-{filename}`
4. **Silver Processing**: All Bronze tables → `Silver.NewCoOrders` 
5. **Gold Aggregation**: Silver data → `Gold.OrderMonthlyKPIs`

### 🚀 Future Enhancements:
- Add data quality validations in Silver
- Implement incremental processing
- Convert to Spark Streaming when volumes increase
- Add error handling and alerting


In [ ]:
# 🔍 Pipeline Verification & Health Check
from pyspark.sql.functions import desc

print("=" * 60)
print("🔍 MEDALLION ARCHITECTURE HEALTH CHECK")
print("=" * 60)

# Check Bronze layer
try:
    bronze_tables = spark.sql("SHOW TABLES IN Bronze").collect()
    print(f"\n🥉 Bronze Layer: {len(bronze_tables)} table(s)")
    for table in bronze_tables:
        table_name = table["tableName"]
        row_count = spark.table(f"Bronze.{table_name}").count()
        print(f"   📊 {table_name}: {row_count:,} records")
except:
    print("\n🥉 Bronze Layer: Not created yet")

# Check Silver layer  
try:
    silver_count = spark.table("Silver.NewCoOrders").count()
    partitions = spark.sql("SHOW PARTITIONS Silver.NewCoOrders").count()
    print(f"\n🥈 Silver Layer:")
    print(f"   📊 Silver.NewCoOrders: {silver_count:,} records")
    print(f"   🗂️  Partitions: {partitions}")
except:
    print("\n🥈 Silver Layer: Not created yet")

# Check Gold layer
try:
    gold_df = spark.table("Gold.OrderMonthlyKPIs")
    total_months = gold_df.select("report_period").distinct().count()
    total_orders = gold_df.agg({"order_count": "sum"}).collect()[0][0]
    print(f"\n🥇 Gold Layer:")
    print(f"   📊 Gold.OrderMonthlyKPIs: {total_months} months")
    print(f"   📈 Total Orders Tracked: {total_orders:,}")
    
    print(f"\n📋 Top Order Statuses:")
    gold_df.groupBy("order_status").agg({"order_count": "sum"}).orderBy(desc("sum(order_count)")).show(5)
except:
    print("\n🥇 Gold Layer: Not created yet")

print("\n✅ Health check complete!")
print("💡 To rerun pipeline: Execute Bronze → Silver → Gold cells in sequence")

StatementMeta(, a639bf75-4ca4-4bfc-b6f3-6c7fb582c1bf, 19, Finished, Available, Finished)

🔍 MEDALLION ARCHITECTURE HEALTH CHECK

🥉 Bronze Layer: 1 table(s)
   📊 orders_olist_orders_dataset_2018: 99,441 records

🥈 Silver Layer: Not created yet

🥇 Gold Layer:
   📊 Gold.OrderMonthlyKPIs: 25 months
   📈 Total Orders Tracked: 99,441

📋 Top Order Statuses:
+------------+----------------+
|order_status|sum(order_count)|
+------------+----------------+
|   delivered|           96478|
|     shipped|            1107|
|    canceled|             625|
| unavailable|             609|
|    invoiced|             314|
+------------+----------------+
only showing top 5 rows


✅ Health check complete!
💡 To rerun pipeline: Execute Bronze → Silver → Gold cells in sequence


In [ ]:
# Quick Silver table verification
print("🔍 Verifying Silver table...")
try:
    silver_tables = spark.sql("SHOW TABLES IN Silver").collect()
    print(f"Silver schema tables: {[t.tableName for t in silver_tables]}")
    
    silver_df = spark.table("Silver.NewCoOrders")
    print(f"Silver.NewCoOrders record count: {silver_df.count():,}")
    print("✅ Silver table verified!")
except Exception as e:
    print(f"❌ Silver verification failed: {e}")

StatementMeta(, a639bf75-4ca4-4bfc-b6f3-6c7fb582c1bf, 20, Finished, Available, Finished)

🔍 Verifying Silver table...
Silver schema tables: ['newcoorders']
Silver.NewCoOrders record count: 99,441
✅ Silver table verified!


Excellent.  I've run the above code a few times and it works flawlessly doing some rudimentary unit tests.    

Let's work on converting this to a Spark Streaming solution.  

Here are the preliminary details I want you to work with.  Remember, ask questions and propose solutions as we go through this.  

Tasks:
* leave the existing code above alone.  
* Create new versions of the code below but modified, with changes annotated, for Spark streaming.  
* I will copy the new code you create below to a notebook specifically for spark streaming. 
* Code should look for new files that exist in "Files/ecommerce/orders-incoming/" for files that have yet to be processed.  
* Process that data to Bronze.orders.  Previously we were using unique table names, instead, I want you to add columns to the table called 
  * "ingestion_timestamp" :  when spark processed the record
  * "ingestion_filename" :  the name of the source csv
* when new data is processed to Bronze.orders then change the necessary code to also stream to Silver and Gold tables already outlined.  


# 🌊 SPARK STREAMING VERSION - NewCo Retail Orders

---

## 🔄 Continuous Processing Architecture 

**From Batch → Streaming Conversion:**
- **File Monitoring**: Auto-detect new CSV files in `Files/ecommerce/orders-incoming/`  
- **Bronze**: Single unified `Bronze.orders` table with metadata tracking  
- **Silver/Gold**: Streaming transformations with continuous updates  
- **Processing**: Near real-time instead of manual cron triggers

### 🚨 Key Architectural Changes:

1. **Bronze Layer**: 
   - ❌ ~~Multiple tables: `Bronze.orders_{filename}`~~  
   - ✅ **Single table**: `Bronze.orders` with `ingestion_filename` column

2. **File Processing**:
   - ❌ ~~Manual file processing~~  
   - ✅ **Auto-discovery** of new files via directory monitoring

3. **Data Flow**:
   - ❌ ~~Batch: Bronze → Silver → Gold sequences~~  
   - ✅ **Streaming**: Continuous Bronze → Silver → Gold pipeline

## 🥉 Bronze Streaming Layer: File Auto-Discovery & Ingestion

**Key Changes from Batch Version:**
- **File Monitoring**: Uses Spark's `readStream` with file source to detect new CSV files  
- **Single Table**: `Bronze.orders` instead of multiple `Bronze.orders_{filename}` tables  
- **Metadata Tracking**: Added `ingestion_filename` column to track source files  
- **Continuous Processing**: Processes files as they arrive

In [ ]:
# 🌊 STREAMING Bronze Layer Implementation
from pyspark.sql.functions import current_timestamp, input_file_name, regexp_extract
from pyspark.sql.types import StructType, StructField, StringType

def setup_bronze_streaming():
    """
    Set up streaming ingestion from file directory to Bronze.orders table
    
    Key Changes from Batch:
    - Uses readStream instead of spark.read 
    - Single Bronze.orders table instead of unique table names
    - Added ingestion_filename column via input_file_name()
    - Continuous processing with writeStream
    """
    
    # Define expected CSV schema (prevents schema evolution issues)
    orders_schema = StructType([
        StructField("order_id", StringType(), True),
        StructField("customer_id", StringType(), True), 
        StructField("order_status", StringType(), True),
        StructField("order_purchase_timestamp", StringType(), True),
        StructField("order_approved_at", StringType(), True),
        StructField("order_delivered_carrier_date", StringType(), True),
        StructField("order_delivered_customer_date", StringType(), True),
        StructField("order_estimated_delivery_date", StringType(), True)
    ])
    
    print("🌊 Setting up Bronze streaming from Files/ecommerce/orders-incoming/")
    
    # Create Bronze schema if it doesn't exist
    spark.sql("CREATE SCHEMA IF NOT EXISTS Bronze")
    
    # Set up file streaming source
    streaming_df = spark.readStream \
        .format("csv") \
        .option("header", "true") \
        .schema(orders_schema) \
        .option("path", "Files/ecommerce/orders-incoming/") \
        .option("maxFilesPerTrigger", 1) \
        .load()
    
    # Add streaming metadata (CHANGED: single table + filename tracking)
    bronze_streaming_df = streaming_df \
        .withColumn("ingestion_timestamp", current_timestamp()) \
        .withColumn("ingestion_filename", 
                   regexp_extract(input_file_name(), r"([^/]+\.csv)$", 1)) \
        .withColumn("processing_date", 
                   current_timestamp().cast("date").cast("string"))
    
    print("✅ Bronze streaming DataFrame configured")
    print("🔄 Key Changes: Single Bronze.orders table + ingestion_filename column")
    
    return bronze_streaming_df

# Configure streaming source
bronze_stream_df = setup_bronze_streaming()

# Display schema to verify structure  
print("\n📋 Bronze Streaming Schema:")
bronze_stream_df.printSchema()

In [ ]:
# 🌊 Start Bronze Streaming Write Operation
def start_bronze_streaming_writer(bronze_df):
    """
    Start the streaming write to Bronze.orders table
    
    Key Changes from Batch:
    - writeStream instead of write
    - append mode for continuous ingestion
    - checkpointing for fault tolerance
    """
    
    print("🚀 Starting Bronze streaming writer...")
    
    bronze_stream_query = bronze_df.writeStream \
        .format("delta") \
        .outputMode("append") \
        .option("checkpointLocation", "/tmp/checkpoints/bronze_orders") \
        .option("mergeSchema", "true") \
        .trigger(processingTime="10 seconds") \
        .toTable("Bronze.orders")
    
    print("✅ Bronze streaming query started!")
    print("🔄 Processing new files every 10 seconds")
    print("📁 Monitoring: Files/ecommerce/orders-incoming/")
    
    return bronze_stream_query

# Start Bronze streaming (uncomment to activate)
# bronze_query = start_bronze_streaming_writer(bronze_stream_df)
print("💡 To start Bronze streaming, uncomment the line above")
print("⚠️  Make sure to add CSV files to Files/ecommerce/orders-incoming/ for testing")

## 🥈 Silver Streaming Layer: Real-time Transformations

**Key Changes from Batch Version:**
- **Streaming Source**: Reads from `Bronze.orders` table as a stream  
- **Continuous Processing**: Transforms data as new records arrive in Bronze  
- **Same Transformations**: Timestamp conversion, partitioning (preserved from batch)  
- **Output**: Continuous updates to `Silver.NewCoOrders`

In [ ]:
# 🌊 STREAMING Silver Layer Implementation  
from pyspark.sql.functions import to_timestamp, year, month, col

def setup_silver_streaming():
    """
    Set up streaming transformation from Bronze.orders to Silver.NewCoOrders
    
    Key Changes from Batch:
    - readStream from Bronze.orders table instead of union of multiple tables
    - Same transformation logic (timestamp conversion, partitioning)
    - Continuous processing instead of one-time batch
    """
    
    print("🥈 Setting up Silver streaming transformations...")
    
    # Create Silver schema if it doesn't exist
    spark.sql("CREATE SCHEMA IF NOT EXISTS Silver")
    
    # Read Bronze table as stream (CHANGED: from single table instead of multiple)
    bronze_stream = spark.readStream \
        .format("delta") \
        .table("Bronze.orders")
    
    # Apply same transformations as batch version
    silver_stream = bronze_stream.select(
        col("order_id"),
        col("customer_id"), 
        col("order_status"),
        # Convert string timestamps to proper timestamp columns (SAME as batch)
        to_timestamp(col("order_purchase_timestamp"), "yyyy-MM-dd HH:mm:ss").alias("order_purchase_timestamp"),
        to_timestamp(col("order_approved_at"), "yyyy-MM-dd HH:mm:ss").alias("order_approved_at"),
        to_timestamp(col("order_delivered_carrier_date"), "yyyy-MM-dd HH:mm:ss").alias("order_delivered_carrier_date"),
        to_timestamp(col("order_delivered_customer_date"), "yyyy-MM-dd HH:mm:ss").alias("order_delivered_customer_date"),
        to_timestamp(col("order_estimated_delivery_date"), "yyyy-MM-dd HH:mm:ss").alias("order_estimated_delivery_date"),
        # Keep metadata columns (CHANGED: now using ingestion_filename)
        col("ingestion_filename"),  # Now tracks individual file names
        col("ingestion_timestamp"), 
        col("processing_date")
    ).withColumn(
        # Add partitioning columns (SAME as batch)
        "order_year", year(col("order_purchase_timestamp"))
    ).withColumn(
        "order_month", month(col("order_purchase_timestamp"))
    )
    
    print("✅ Silver streaming DataFrame configured")
    print("🔄 Key Changes: Reading from single Bronze.orders table")
    
    return silver_stream

def start_silver_streaming_writer(silver_df):
    """
    Start the streaming write to Silver.NewCoOrders table
    """
    
    print("🚀 Starting Silver streaming writer...")
    
    silver_stream_query = silver_df.writeStream \
        .format("delta") \
        .outputMode("append") \
        .option("checkpointLocation", "/tmp/checkpoints/silver_newco_orders") \
        .option("mergeSchema", "true") \
        .partitionBy("order_year", "order_month") \
        .trigger(processingTime="15 seconds") \
        .toTable("Silver.NewCoOrders")
    
    print("✅ Silver streaming query started!")
    print("📊 Continuously transforming Bronze → Silver")
    
    return silver_stream_query

# Configure Silver streaming 
silver_stream_df = setup_silver_streaming()

# Start Silver streaming (uncomment to activate)
# silver_query = start_silver_streaming_writer(silver_stream_df) 
print("💡 To start Silver streaming, uncomment the line above")

## 🥇 Gold Streaming Layer: Real-time KPI Updates

**Key Changes from Batch Version:**
- **Streaming Aggregation**: Continuous updates to monthly KPIs as new orders arrive  
- **Stateful Processing**: Maintains monthly counts that update when new data arrives  
- **Complete Output Mode**: Rewrites entire monthly aggregations (ensures accuracy)  
- **Near Real-time**: KPIs available within seconds of new order processing

**⚠️ Design Decision**: Using `complete` output mode to ensure accurate monthly counts as new orders can affect historical months (late arrivals, status updates).

In [ ]:
# 🌊 STREAMING Gold Layer Implementation
from pyspark.sql.functions import count, desc, concat, lpad, lit

def setup_gold_streaming():
    """
    Set up streaming aggregation from Silver.NewCoOrders to Gold.OrderMonthlyKPIs
    
    Key Changes from Batch:
    - readStream from Silver.NewCoOrders instead of static read
    - Stateful streaming aggregation (maintains running counts) 
    - complete output mode for accurate monthly totals
    - Continuous KPI updates instead of batch recalculation
    """
    
    print("🥇 Setting up Gold streaming aggregations...")
    
    # Create Gold schema if it doesn't exist
    spark.sql("CREATE SCHEMA IF NOT EXISTS Gold")
    
    # Read Silver table as stream (CHANGED: stream instead of batch)
    silver_stream = spark.readStream \
        .format("delta") \
        .table("Silver.NewCoOrders")
    
    # Create streaming aggregations (SAME logic as batch, but streaming)
    monthly_kpis_stream = silver_stream.groupBy("order_year", "order_month", "order_status") \
                                    .agg(count("order_id").alias("order_count")) \
                                    .withColumn("report_period", 
                                              concat(col("order_year").cast("string"), 
                                                    lit("-"),
                                                    lpad(col("order_month").cast("string"), 2, "0"))) \
                                    .orderBy("order_year", "order_month", desc("order_count"))
    
    print("✅ Gold streaming DataFrame configured")
    print("🔄 Key Changes: Stateful aggregation with continuous updates")
    
    return monthly_kpis_stream

def start_gold_streaming_writer(gold_df):
    """
    Start the streaming write to Gold.OrderMonthlyKPIs table
    
    Note: Using complete output mode to ensure accurate monthly totals
    """
    
    print("🚀 Starting Gold streaming writer...")
    
    gold_stream_query = gold_df.writeStream \
        .format("delta") \
        .outputMode("complete") \
        .option("checkpointLocation", "/tmp/checkpoints/gold_monthly_kpis") \
        .option("mergeSchema", "true") \
        .trigger(processingTime="30 seconds") \
        .toTable("Gold.OrderMonthlyKPIs")
    
    print("✅ Gold streaming query started!")
    print("📈 Continuously updating monthly KPIs")
    print("⚠️  Using 'complete' mode - rewrites entire table for accuracy")
    
    return gold_stream_query

# Configure Gold streaming
gold_stream_df = setup_gold_streaming()

# Start Gold streaming (uncomment to activate)  
# gold_query = start_gold_streaming_writer(gold_stream_df)
print("💡 To start Gold streaming, uncomment the line above")

## 🔀 Complete Streaming Pipeline Orchestration

**End-to-End Streaming Flow:**
1. **File Drop** → `Files/ecommerce/orders-incoming/` *(New CSV files arrive)*  
2. **Bronze Stream** → Auto-detects files → `Bronze.orders` *(10 sec trigger)*  
3. **Silver Stream** → Transforms Bronze → `Silver.NewCoOrders` *(15 sec trigger)*  
4. **Gold Stream** → Aggregates Silver → `Gold.OrderMonthlyKPIs` *(30 sec trigger)*

**🚀 Single Command Pipeline Start:**

In [ ]:
# 🚀 Complete Streaming Pipeline - Start All Layers
def start_complete_streaming_pipeline():
    """
    Start the complete Bronze → Silver → Gold streaming pipeline
    
    Returns: Dictionary of streaming query objects for monitoring
    """
    
    print("🔥 STARTING COMPLETE NEWCO RETAIL STREAMING PIPELINE")
    print("=" * 60)
    
    # Initialize all streams
    bronze_df = setup_bronze_streaming()
    silver_df = setup_silver_streaming() 
    gold_df = setup_gold_streaming()
    
    # Start all streaming queries
    print("\n1️⃣ Starting Bronze streaming (Files → Bronze.orders)...")
    bronze_query = start_bronze_streaming_writer(bronze_df)
    
    print("\n2️⃣ Starting Silver streaming (Bronze → Silver.NewCoOrders)...")
    silver_query = start_silver_streaming_writer(silver_df)
    
    print("\n3️⃣ Starting Gold streaming (Silver → Gold.OrderMonthlyKPIs)...")
    gold_query = start_gold_streaming_writer(gold_df)
    
    print("\n🎉 COMPLETE STREAMING PIPELINE ACTIVE!")
    print("📂 Drop CSV files into Files/ecommerce/orders-incoming/")
    print("⏱️  Processing triggers: Bronze(10s) → Silver(15s) → Gold(30s)")
    
    # Return query objects for monitoring
    return {
        "bronze": bronze_query,
        "silver": silver_query, 
        "gold": gold_query
    }

# Complete pipeline starter (UNCOMMENT TO ACTIVATE ALL)
# streaming_queries = start_complete_streaming_pipeline()

print("💡 To start complete streaming pipeline:")
print("   🔓 Uncomment the line above")
print("   📁 Add CSV files to Files/ecommerce/orders-incoming/") 
print("   ⏰ Watch real-time processing!")

## 📊 Streaming Pipeline Monitoring & Management

**Real-time Monitoring Tools:**

In [ ]:
# 📊 Streaming Pipeline Monitoring Tools

def monitor_streaming_health():
    """
    Monitor all active streaming queries and table health
    """
    
    print("🔍 STREAMING PIPELINE HEALTH CHECK")
    print("=" * 50)
    
    # Check active streaming queries
    print("\n💫 Active Streaming Queries:")
    active_streams = spark.streams.active
    if active_streams:
        for i, stream in enumerate(active_streams, 1):
            print(f"   {i}. {stream.name or 'Unnamed'} - Status: {stream.lastProgress}")
    else:
        print("   ⭕ No active streaming queries")
    
    # Check table row counts  
    print("\n📋 Table Health:")
    tables_to_check = [
        "Bronze.orders",
        "Silver.NewCoOrders", 
        "Gold.OrderMonthlyKPIs"
    ]
    
    for table in tables_to_check:
        try:
            count = spark.table(table).count()
            print(f"   ✅ {table}: {count:,} records")
        except:
            print(f"   ❌ {table}: Table not found or empty")

def stop_all_streaming_queries():
    """
    Stop all active streaming queries gracefully
    """
    
    print("🛑 Stopping all streaming queries...")
    
    active_streams = spark.streams.active
    if active_streams:
        for stream in active_streams:
            stream.stop()
            print(f"   ⏹️  Stopped: {stream.name or 'Unnamed'}")
        print("✅ All streams stopped successfully")
    else:
        print("   ℹ️  No active streams to stop")

def test_file_processing(test_filename="test_orders_sample.csv"):
    """
    Create a test CSV file to verify streaming processing
    """
    
    print(f"📝 Creating test file: {test_filename}")
    
    # Create sample order data
    test_data = """order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
test001,cust001,delivered,2024-02-18 10:30:00,2024-02-18 10:35:00,2024-02-19 14:20:00,2024-02-21 16:45:00,2024-02-25 00:00:00
test002,cust002,shipped,2024-02-18 11:15:00,2024-02-18 11:20:00,2024-02-19 09:30:00,,2024-02-26 00:00:00"""
    
    # Write test file (Note: This is demonstration - actual file writing depends on your environment)
    print("📁 Test data preview:")
    print(test_data)
    print("💡 Copy this data to a CSV file in Files/ecommerce/orders-incoming/ to test streaming!")

# Run health check
monitor_streaming_health()

# Uncomment to create test data
# test_file_processing()

## 📋 Batch vs Streaming: Key Architectural Changes

| **Component** | **🏠 Batch Version** | **🌊 Streaming Version** | **Key Changes** |
|---------------|-------------------|----------------------|----------------|
| **Bronze Tables** | Multiple: `Bronze.orders_{filename}` | Single: `Bronze.orders` | ✅ Unified table + `ingestion_filename` column |
| **File Processing** | Manual: Load specific files | Auto: Monitor directory | ✅ File system watching with `readStream` |
| **Data Flow** | Sequential: Run cells manually | Continuous: Always running | ✅ Real-time processing with triggers |
| **Silver Processing** | Union multiple Bronze tables | Stream from single Bronze table | ✅ Simplified source reading |  
| **Gold KPIs** | Batch recalculation | Stateful streaming aggregation | ✅ Incremental monthly counts |
| **Fault Tolerance** | Manual rerun if failure | Checkpointing + auto-recovery | ✅ Built-in resilience |

## 🎯 When to Use Each Approach:

**👍 Use Batch Version When:**
- Processing historical data loads  
- Running ad-hoc analysis  
- File drops are infrequent (daily/weekly)
- Manual control preferred

**🚀 Use Streaming Version When:**
- Files arrive frequently (hourly/continuous)  
- Near real-time KPIs required
- Automated processing needed
- High availability requirements

## ⚠️ Streaming Considerations & Best Practices:

### 📁 File Management:
- **Move processed files**: Prevent reprocessing by moving files out of watch directory
- **File naming**: Use timestamp prefixes for processing order
- **Schema evolution**: Test new CSV schemas carefully

### 💾 Resource Management:
- **Checkpoint cleanup**: Periodically clean `/tmp/checkpoints/` directories
- **Memory usage**: Monitor Spark driver memory for large aggregations
- **Parallelism**: Tune `maxFilesPerTrigger` based on file sizes

### 🛠️ Production Deployment:
- **Error handling**: Add dead letter queues for malformed files
- **Alerting**: Monitor stream health and data freshness  
- **Scaling**: Consider cluster auto-scaling for variable loads


## 🚀 Quick Start Guide for Streaming Version

### **Step 1: Test with Existing Data**
1. Copy your current CSV to `Files/ecommerce/orders-incoming/`  
2. Run: `streaming_queries = start_complete_streaming_pipeline()`  
3. Monitor: `monitor_streaming_health()`  

### **Step 2: Production Deployment**  
1. **Copy streaming code** to dedicated notebook  
2. **Configure file management** (move processed files)  
3. **Set up monitoring** alerts for stream failures  
4. **Test with sample data** before full deployment  

### **Step 3: Scale & Optimize**
1. **Tune trigger intervals** based on data volume  
2. **Implement dead letter queues** for error handling  
3. **Add data quality checks** in streaming transformations  
4. **Monitor resource usage** and scale cluster as needed  

---

## 💡 Questions for Implementation Discussion:

1. **File Management**: How should we handle processed files? (Move, archive, delete?)
2. **Error Handling**: What should happen with malformed CSV files?  
3. **Late Data**: How should we handle orders with past timestamps?
4. **Alerting**: What metrics should trigger operational alerts?  
5. **Backup Strategy**: Should we maintain batch processing as backup?

## 🎯 Next Steps:
- [ ] Test streaming code with sample files  
- [ ] Implement error handling and dead letter queue  
- [ ] Set up monitoring and alerting  
- [ ] Create production deployment plan  
- [ ] Define file management strategy

**✅ Your streaming medallion architecture for NewCo Retail is ready for deployment!**


# What is next?  How can we make this better?  

* (wip)ghcp/vscode with `plan` mode and `agent` mode:  
  * work through an example with ghcp.  Start with plan mode.  This way it can ask questions to refine the sares it does not understand.  Use opus 4.6 for this.  Once the plan is done move into agent mode and have it create one based on the plan.  Iterate here to get the best result.  
  * once this hits 80%ish outcome, tell it to make this a skill.  then when calling the skill you can specify the last 20%.  

or

* do the basic EDA
* feed that code to ghcp with a basic streaming template


## Functional Requirements:  What needs to be in the streaming template?  

* sections to handle the basic workflow, ie
  * create object section:  create objects _if not exists_ style
  * read data:  with ability to stream/batch, if stream, pass in the HWM/replay/reprocess
  * DQ/fixup data section
  * business logic section
  * write data section
  * system resetter/monitor/metrics/observability section
  * unit tests
* the ability to enable/disable sections as needed
* **the ability to customize the template based on the customer's culture, coding standards, etc.  (like constitution.md in speckit)
* common workflow "gotcha" scenarios with spark streaming
  * Prompt similar to this maybe:  ` "the pipeline failed.  Make a copy of the notebook, put spark streaming in debug mode but do not write the data out, help me find the error, fix the error, pause all streaming and note the timestamp of the failure and when you paused it, apply the fix to the original notebook, restart the stream BUT mark the .startAT as 5 mins before the failure, just to be safe."  `


## NFRs
* observability section
* installability, rerunnability, FinOps stuff, etc




